In [87]:
import os
import polars as pl
import altair as alt

In [88]:
pl.Config(tbl_rows=100)
pl.Config(fmt_str_lengths=150)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [89]:
df = pl.read_parquet(
    "data/segments.parquet"
).sort(
    ["id", "date"]
).with_columns(
    days_diff = pl.col("date").diff().dt.total_days().over("id"),
    effort_diff = pl.col("effort_count").diff().over("id")
).with_columns(
    avg_daily_efforts = pl.col("effort_diff") / pl.col("days_diff")
).with_columns(
    pl.when(pl.col("city").str.contains("Praha") | pl.col("city").str.contains("Prague")).then(pl.lit('Praha')).when(pl.col('city').str.contains('Brno')).then(pl.lit('Brno')).otherwise(pl.col('city')).alias('city'),
    pl.when(pl.col("effort_diff") < 0).then(pl.lit(0)).otherwise(pl.col("effort_diff")).alias("effort_diff")
).with_columns(
    pl.col('date').dt.ordinal_day().alias('day'),
    pl.col('date').dt.week().alias('week'),
    pl.col('date').dt.year().alias('year')
)

## Rozsah dat

In [91]:
print(df.select(pl.col("date").min()).item())
print(df.select(pl.col("date").max()).item())

2024-12-24 23:05:01+01:00
2026-03-08 23:39:05+01:00


In [92]:
df.sample(1)

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance,days_diff,effort_diff,avg_daily_efforts,day,week,year
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64,i64,i64,f64,i16,i8,i32
11655930,"""Mezi zábranama (proti proudu)""","""Run""",1067.3,-0.2,255.0,250.8,3.5,"[50.344875, 15.929941]","[50.354198, 15.931006]","""Czechia""","""Hradec Králové Region""","""Jaroměř""",1509,138,2025-01-01 23:05:35 CET,1.039421,1,0,0.0,1,1,2025


## Výběr segmentů ke sledování

In [94]:
zebricek = df.filter(
    pl.col("country") == "Czechia"
).group_by(
    ["name","city","activity_type","start_to_finish_distance"]
).agg(
    pl.col("effort_diff").quantile(0.8)
).sort(
    by="effort_diff", descending=True
)

zebricek.head(100)

name,city,activity_type,start_to_finish_distance,effort_diff
str,str,str,f64,f64
"""Die Eine Runde""","""Brno""","""Ride""",0.002446,414.0
"""VUT horní dráha 400m""","""Brno""","""Run""",0.011266,219.0
"""Opičí Časovka, Podolská vodárna - Dvorce""","""Praha""","""Ride""",1.379737,210.0
"""Zweite hup u ZOO""","""Praha""","""Ride""",0.308372,188.0
"""Stadion Na Děkance""","""Praha""","""Run""",0.006087,187.0
"""Zbraslav - Komořany""","""Zbraslav""","""Ride""",2.258969,151.0
"""Pražačka 320""","""Praha""","""Run""",0.000925,140.0
"""Šlechtovka - finální rovinka""","""Praha""","""Run""",0.222849,139.0
"""Lužánky rovinka""","""Brno""","""Run""",0.260571,124.0


In [95]:
celorepublika = zebricek.filter(
    (pl.col('activity_type') == 'Run') & (pl.col('start_to_finish_distance') > 0.5)
).unique(
    subset='city',keep='first'
).sort(
    by='effort_diff',descending=True
).head(
    20
).select(
    pl.col('name')
).to_series(
).to_list()

celorepublika

['tenis 1K',
 'Smetanovy reverse',
 'Hvězda flat',
 'Stezka nábřeží',
 'stezka',
 'Hradní lávka => Sýkorův most',
 'mezi mosty',
 'Panelovka-Tichá 800m',
 'Ku řopíku',
 'Od Sanusu ke gymplu',
 'Lípy (směrem do Jičína)',
 'od hráze až na konec přehrady - silnice',
 'U Jordánu do kopce',
 'Tuhnická lávka -Chebský most',
 'Nábřeží',
 'po cyklostezce',
 'Ciclo Via BNL -> K',
 'Dlouhá míle',
 'Kuželna - Splav 800m',
 'BK-nabrezi']

In [96]:
import yaml

In [97]:
with open("data_raw/behani_top_20.yaml", "w", encoding="utf-8") as f:
    yaml.dump(celorepublika, f, allow_unicode=True, default_flow_style=False)

In [98]:
celorepublika_kolo = zebricek.filter(
    (pl.col('activity_type') == 'Ride') & (pl.col('start_to_finish_distance') > 1)
).unique(
    subset='city',keep='first'
).sort(
    by='effort_diff',descending=True
).head(
    20
).select(
    pl.col('name')
).to_series(
).to_list()

celorepublika_kolo

['Opičí Časovka, Podolská vodárna - Dvorce',
 'Zbraslav - Komořany',
 'time trial at Svratka',
 'Stezka do Zlina',
 'Židlochovice -> Olympia',
 'Srbsko - Hlásná Třebáň',
 'Revnicak Eve',
 'Muglinovská→Hradní lávka (po pravém břehu)',
 'Novomlýnská 5: Věstonice-Strachotín',
 'svadov—valtirov',
 'Jiráskovo nábřeží k Husovce',
 'Cyklostezka sprint směr Děčín',
 'Nová cyklostezka do Bosche  2',
 'Maslovice - Kralupy',
 'Sprintík křižovatka/skalní mlýn',
 'Adamaov climb',
 'Trinec-Tesin',
 'Szpic Jested  last 3 km ( od rozc na Dub)',
 'Lysá hora - 5 km',
 'Mutěnka - z Kyjova po odbočku na hraběcí']

In [99]:
with open("data_raw/kolo_top_20.yaml", "w", encoding="utf-8") as f:
    yaml.dump(celorepublika_kolo, f, allow_unicode=True, default_flow_style=False)

In [100]:
df.filter(
    pl.col('name').is_in(celorepublika)
)

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance,days_diff,effort_diff,avg_daily_efforts,day,week,year
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64,i64,i64,f64,i16,i8,i32
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49709,3096,2024-12-24 23:05:06 CET,0.995211,null,null,null,359,52,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49718,3097,2024-12-25 23:35:53 CET,0.995211,1,9,9.0,360,52,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49720,3098,2024-12-26 23:21:59 CET,0.995211,0,2,inf,361,52,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49731,3099,2024-12-27 23:38:17 CET,0.995211,1,11,11.0,362,52,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49750,3100,2024-12-28 23:38:27 CET,0.995211,1,19,19.0,363,52,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49766,3100,2024-12-29 23:38:38 CET,0.995211,1,16,16.0,364,52,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49771,3104,2024-12-30 23:22:09 CET,0.995211,0,5,inf,365,1,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49789,3109,2024-12-31 23:22:07 CET,0.995211,0,18,inf,366,1,2024
2462668,"""Hvězda flat""","""Run""",1001.15,0.6,383.8,372.4,11.426,"[50.083141, 14.32784]","[50.081176, 14.341448]","""Czechia""","""Hlavní město Praha""","""Praha""",49828,3110,2025-01-01 23:22:08 CET,0.995211,1,39,39.0,1,1,2025


In [101]:
orez_extremu = df.filter(
    pl.col('name').is_in(celorepublika)
).select(
    pl.col('effort_diff')
).quantile(
    0.99
).item(
)

do_grafu_celorepublika = df.filter(
    pl.col('name').is_in(celorepublika)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu).then(orez_extremu).otherwise(pl.col('effort_diff')).alias('effort_diff')
).group_by(
    ['year','week']
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['year','week']
).filter(
    pl.col('year') >= 2025
).with_columns(
    (pl.col('effort_diff') / pl.col('effort_diff').shift(1)).alias('trend')
)

do_grafu_celorepublika.filter(pl.col("trend") > 1.2)

year,week,effort_diff,trend
i32,i8,f64,f64
2025,44,1460.0,1.418853
2025,47,1615.0,1.78453
2026,2,1024.0,1.410468
2026,6,1348.0,1.454153
2026,8,1188.0,1.346939
2026,9,1589.0,1.337542


In [102]:
orez_extremu_kolo = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).select(
    pl.col('effort_diff')
).quantile(
    0.99
).item(
)

do_grafu_celorepublika_kolo = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu_kolo).then(orez_extremu_kolo).otherwise(pl.col('effort_diff')).alias('effort_diff')
).group_by(
    ['year','week']
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['year','week']
).filter(
    pl.col('year') >= 2025
).with_columns(
    (pl.col('effort_diff') / pl.col('effort_diff').shift(1)).alias('trend')
)

do_grafu_celorepublika_kolo.filter(pl.col("trend") > 1.2)

do_grafu_celorepublika_kolo_2w = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu_kolo).then(orez_extremu_kolo).otherwise(pl.col('effort_diff')).alias('effort_diff')
).sort(
    by="date"
).group_by_dynamic(
    index_column="date", every="2w"
).agg(
    pl.col('effort_diff').sum()
).sort(
    by="date"
).filter(
    pl.col('date').dt.year() >= 2025
).with_columns(
    (pl.col('effort_diff') / pl.col('effort_diff').shift(1)).alias('trend')
).with_columns(
    pl.col("date").dt.year().alias("year")
)

do_grafu_celorepublika_kolo.filter(pl.col("trend") > 1.2)

year,week,effort_diff,trend
i32,i8,f64,f64
2025,3,761.0,1.206022
2025,4,1419.0,1.864652
2025,8,1169.0,1.562834
2025,9,2075.0,1.775021
2025,10,3980.0,1.918072
2025,12,3192.0,1.479833
2025,16,4825.0,1.204744
2025,22,5643.0,1.223282
2025,24,6543.0,1.456589


In [103]:
do_grafu_celorepublika.sort(by="effort_diff",descending=True).head(10)

year,week,effort_diff,trend
i32,i8,f64,f64
2026,10,1619.0,1.01888
2025,47,1615.0,1.78453
2026,9,1589.0,1.337542
2025,36,1462.0,1.160317
2025,44,1460.0,1.418853
2025,22,1433.0,1.166938
2025,19,1428.0,1.088415
2025,50,1423.0,1.117832
2025,23,1417.0,0.988835


In [104]:
alt.Chart(
    do_grafu_celorepublika,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('trend:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [105]:
alt.Chart(
    do_grafu_celorepublika_kolo,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('trend:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [106]:
do_grafu_celorepublika_kolo_2w

date,effort_diff,trend,year
"datetime[μs, CET]",f64,f64,i32
2025-01-06 00:00:00 CET,1392.0,null,2025
2025-01-20 00:00:00 CET,3029.0,2.176006,2025
2025-02-03 00:00:00 CET,1961.0,0.647408,2025
2025-02-17 00:00:00 CET,3244.0,1.654258,2025
2025-03-03 00:00:00 CET,6137.0,1.8918,2025
2025-03-17 00:00:00 CET,6374.0,1.038618,2025
2025-03-31 00:00:00 CEST,7801.0,1.223878,2025
2025-04-14 00:00:00 CEST,9914.0,1.270863,2025
2025-04-28 00:00:00 CEST,10756.0,1.08493,2025


In [107]:
alt.Chart(
    do_grafu_celorepublika_kolo_2w,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('date:T'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [108]:
alt.Chart(
    do_grafu_celorepublika,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [109]:
do_grafu_celorepublika

year,week,effort_diff,trend
i32,i8,f64,f64
2025,1,1085.0,null
2025,2,1179.0,1.086636
2025,3,1128.0,0.956743
2025,4,1250.0,1.108156
2025,5,1235.0,0.988
2025,6,1040.0,0.842105
2025,7,1053.0,1.0125
2025,8,1120.0,1.063628
2025,9,1170.0,1.044643


In [177]:
alt.Chart(
    do_grafu_celorepublika,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [181]:
do_grafu_celorepublika.drop("trend").write_json("data/graf_rok_behani.json")

In [110]:
alt.Chart(
    do_grafu_celorepublika_kolo,
    width=500,
    height=300
).mark_line(
).encode(
    alt.X('week:Q'),
    alt.Y('effort_diff:Q'),
    alt.Color('year:N')
)

alt.Chart(...)

In [183]:
do_grafu_celorepublika_kolo.drop("trend").write_json("data/graf_rok_cyklo.json")

In [111]:
df.filter(
    (pl.col("country") == "Czechia") & (pl.col("activity_type") == "Run")
).group_by(
    "city"
).agg(
    pl.col("name").unique().len()
).sort(
    by='name',descending=True
)

city,name
str,u32
"""Praha""",16
"""Brno""",12
"""Ostrava""",4
"""Olomouc""",3
"""Hradec Králové""",3
"""Liberec""",3
"""Jihlava""",2
"""Klatovy""",2
"""Sušice""",2


In [112]:
df.filter(
    (pl.col("country") == "Czechia") & (pl.col("activity_type") == "Ride") & (pl.col('start_to_finish_distance') > 1)
).group_by(
    ["name","city"]
).agg(
    pl.col("effort_diff").quantile(0.8)
).sort(
    by="effort_diff", descending=True
)

name,city,effort_diff
str,str,f64
"""Opičí Časovka, Podolská vodárna - Dvorce""","""Praha""",210.0
"""Zbraslav - Komořany""","""Zbraslav""",151.0
"""time trial at Svratka""","""Brno""",92.0
"""Rokytka od myčky""","""Praha""",66.0
"""Stezka do Zlina""","""Zlin""",51.0
"""Židlochovice -> Olympia""","""Židlochovice""",32.0
"""Srbsko - Hlásná Třebáň""","""Srbsko""",29.0
"""Revnicak Eve""","""Řevnice""",28.0
"""Muglinovská→Hradní lávka (po pravém břehu)""","""Ostrava""",28.0


In [113]:
df

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance,days_diff,effort_diff,avg_daily_efforts,day,week,year
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64,i64,i64,f64,i16,i8,i32
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336364,152211,2024-12-27 23:05:12 CET,3.422275,null,null,null,362,52,2024
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336351,152210,2024-12-28 23:38:27 CET,3.422275,1,0,-13.0,363,52,2024
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336355,152208,2024-12-29 23:38:28 CET,3.422275,1,4,4.0,364,52,2024
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336357,152205,2024-12-30 23:05:20 CET,3.422275,0,2,inf,365,1,2024
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336362,152204,2024-12-31 23:05:30 CET,3.422275,1,5,5.0,366,1,2024
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336345,152202,2025-01-01 23:22:12 CET,3.422275,1,0,-17.0,1,1,2025
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336344,152200,2025-01-02 23:38:22 CET,3.422275,1,0,-1.0,2,1,2025
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336346,152202,2025-01-03 23:05:25 CET,3.422275,0,2,inf,3,1,2025
652851,"""Alpe d'Huez""","""Ride""",12024.9,8.8,1775.7,728.4,1047.3,"[45.064932, 6.039987]","[45.090662, 6.063903]","""France""","""Rhône-Alpes""","""Le Bourg D'Oisans""",336354,152203,2025-01-04 23:22:00 CET,3.422275,1,8,8.0,4,1,2025


In [114]:
df.select("effort_diff").quantile(0.9)

effort_diff
f64
28.0


In [115]:
df.filter(
    (pl.col("country") == "Czechia") & (pl.col("activity_type") == "Run") & (pl.col('start_to_finish_distance') < 0.5)
).group_by(
    ["name","city"]
).agg(
    pl.col("effort_diff").quantile(0.8)
).sort(
    by="effort_diff", descending=True
)

name,city,effort_diff
str,str,f64
"""VUT horní dráha 400m""","""Brno""",219.0
"""Stadion Na Děkance""","""Praha""",187.0
"""Pražačka 320""","""Praha""",140.0
"""Šlechtovka - finální rovinka""","""Praha""",139.0
"""Lužánky rovinka""","""Brno""",124.0
"""TJ Sokol okruh""","""Praha""",91.0
"""400m-Liberec""","""Liberec""",74.0
"""Strahovská dráha""","""Praha""",66.0
"""Štvanická lávka (do Karlína)""","""Praha""",64.0


In [116]:
df.filter(
    (pl.col("country") == "Czechia") & (pl.col("activity_type") == "Run") & (pl.col('start_to_finish_distance') > 0.5)
).group_by(
    ["name","city"]
).agg(
    pl.col("effort_diff").quantile(0.8)
).sort(
    by="effort_diff", descending=True
)

name,city,effort_diff
str,str,f64
"""tenis 1K""","""Brno""",45.0
"""Smetanovy reverse""","""Olomouc""",41.0
"""Hvězda flat""","""Praha""",33.0
"""Stodůlecký - Nepomucký rybník""","""Praha""",30.0
"""Stezka nábřeží""","""Zlín""",22.0
"""Rullerovo nábřeží""","""Brno""",20.0
"""stezka""","""České Budějovice""",17.0
"""Hradní lávka => Sýkorův most""","""Ostrava""",14.0
"""mezi mosty""","""Pardubice""",13.0


In [117]:
alt.Chart(
    df.filter(
        pl.col('date').dt.year() >= 2025
    ).filter(
        pl.col('name').is_in(
            ["Opičí Časovka, Podolská vodárna - Dvorce","time trial at Svratka","Muglinovská→Hradní lávka (po pravém břehu)"	]
        )
    ).sort(
        by='date'
    ).group_by_dynamic(
        index_column='date',every='1mo'
    ).agg(
        pl.col('effort_diff').sum()
    ),
    width=800
).mark_bar(
).encode(
    alt.X("date:T"),
    alt.Y('effort_diff:Q')
    )

alt.Chart(...)

In [118]:
alt.Chart(
    df.filter(
        pl.col('date').dt.year() >= 2025
    ).filter(
        pl.col('name').is_in(
            ["Šlechtovka - finální rovinka","Lužánky rovinka","Smetanovy reverse","tenis 1K","Hvězda flat"]
        )
    ).sort(
        by='date'
    ).group_by_dynamic(
        index_column='date',every='1mo'
    ).agg(
        pl.col('effort_diff').sum()
    ),
    width=800
).mark_bar(
).encode(
    alt.X("date:T"),
    alt.Y('effort_diff:Q')
    )

alt.Chart(...)

In [119]:
alt.Chart(
    df.filter(
        pl.col('date').dt.year() >= 2025
    ).filter(
        pl.col('name').is_in(
            ["Die Eine Runde"	]
        )
    ).sort(
        by='date'
    ).group_by_dynamic(
        index_column='date',every='1mo'
    ).agg(
        pl.col('effort_diff').sum()
    ),
    width=800
).mark_bar(
).encode(
    alt.X("date:T"),
    alt.Y('effort_diff:Q')
    )

alt.Chart(...)

In [120]:
def ukaz_mesice(segmenty:list):
    return alt.Chart(
    df.filter(
        pl.col('date').dt.year() >= 2025
    ).filter(
        pl.col('name').is_in(
            segmenty
        )
    ).sort(
        by='date'
    ).group_by_dynamic(
        index_column='date',every='1mo'
    ).agg(
        pl.col('effort_diff').sum()
    ),
    width=800
).mark_bar(
).encode(
    alt.X("date:T"),
    alt.Y('effort_diff:Q')
    )

In [121]:
ukaz_mesice(
    ["Adamaov climb"	]
)

alt.Chart(...)

In [122]:
ukaz_mesice(["VUT horní dráha 400m","Stadion Na Děkance","TJ Sokol okruh","400m-Liberec","Kolo na stadionu v Třebíči"])

alt.Chart(...)

In [123]:
def ukaz_dny(segmenty:list):
    return alt.Chart(
    df.filter(
        pl.col('date').dt.year() >= 2025
    ).filter(
        pl.col('name').is_in(
            segmenty
        )
    ).sort(
        by='date'
    ).group_by(
        pl.col("date").dt.weekday()
    ).agg(
        pl.col('effort_diff').mean()
    ),
    width=800
).mark_bar(
).encode(
    alt.X("date:N"),
    alt.Y('effort_diff:Q')
    )

In [124]:
ukaz_dny(["Rullerovo nábřeží"])

alt.Chart(...)

In [125]:
ukaz_dny(["VUT horní dráha 400m","Stadion Na Děkance","TJ Sokol okruh","400m-Liberec","Kolo na stadionu v Třebíči"])

alt.Chart(...)

In [126]:
ukaz_dny(
    ["Kleť k vysílači - finále","Lednice - Bulhary","Máchovo kritérium - přes bory kolem vrchu Borný","Špindlerovka - část II. (od Černýho potoka nahoru)"]
)

alt.Chart(...)

In [127]:
sledujeme = [
    "Šlechtovka - finální rovinka","Štvanická lávka (do Karlína)","Most Legií (směr Kampa)","Vítkov up sprint","Hvězda flat","Šárka Valley (short)","Stodůlecký - Nepomucký rybník"
]

In [128]:
ukaz_dny(
    sledujeme
)

alt.Chart(...)

In [129]:
mezirocni_srovnani = df.filter(
    pl.col("name").is_in(sledujeme) & (pl.col('date').dt.year() >= 2025)
).with_columns(
    pl.col("date").dt.year().alias("rok"),
    pl.col("date").dt.ordinal_day().alias("den"),
    pl.col("date").dt.week().alias("tyden"),
).group_by(
    ['rok','tyden']
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['rok','tyden']
)

strop = mezirocni_srovnani.select(pl.col('effort_diff')).quantile(0.97).item()

mezirocni_srovnani = mezirocni_srovnani.with_columns(
    pl.when(pl.col('effort_diff') > strop).then(pl.lit(strop)).otherwise(pl.col('effort_diff')).alias('effort_diff')
)

In [130]:
alt.Chart(
    mezirocni_srovnani,
    width=900
).mark_line(
).encode(
    alt.X('tyden:Q'),
    alt.Y('effort_diff:Q'),
    alt.Color('rok:N')
)

alt.Chart(...)